So we want to construct the csv like we did previously but this time we only want to consider the leaf nodes of each questions and then add them in as as a skill in the column vector. now we name each column vector after the name of the skill. Note there should only be columns from the trees

# we want to build a csv with the following columns:
# QuestionId,
# UserId
# AnswerId
# IsCorrect
# TimeTaken for the student to complete the question
# All of the subject metadata file in the /data/metadata/ folder titled subject_metadata.csv

Be sure to include a percentage bar of how much the programm is complete that regurally updates so i can see if the program paused while runnnign.

In [1]:
from __future__ import annotations

import ast
import csv
import sys
from pathlib import Path
from typing import Iterable


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    candidates = [current, *current.parents]
    for p in candidates:
        if (p / "data" / "metadata" / "subject_metadata.csv").exists():
            return p
    raise FileNotFoundError("Could not locate project root containing data/metadata/subject_metadata.csv")


def parse_subject_chain(raw: str) -> list[int]:
    try:
        values = ast.literal_eval(raw)
        if isinstance(values, list):
            return [int(x) for x in values]
    except (ValueError, SyntaxError):
        pass
    return []


def print_progress(current: int, total: int, *, prefix: str = "Progress", width: int = 40) -> None:
    if total <= 0:
        return
    frac = min(max(current / total, 0.0), 1.0)
    filled = int(width * frac)
    bar = "#" * filled + "-" * (width - filled)
    percent = frac * 100
    sys.stdout.write(f"\r{prefix}: [{bar}] {percent:6.2f}% ({current}/{total})")
    sys.stdout.flush()
    if current >= total:
        sys.stdout.write("\n")


def count_data_rows(csv_path: Path) -> int:
    with csv_path.open("r", encoding="utf-8-sig", newline="") as f:
        # subtract 1 for header row
        return max(sum(1 for _ in f) - 1, 0)


def load_subject_metadata(subject_csv: Path) -> tuple[dict[int, str], set[int]]:
    subject_name_by_id: dict[int, str] = {}
    parent_ids: set[int] = set()
    all_ids: set[int] = set()

    with subject_csv.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            sid = int(row["SubjectId"])
            all_ids.add(sid)
            subject_name_by_id[sid] = row["Name"].strip()

            parent = row.get("ParentId", "")
            if parent and parent != "NULL":
                parent_ids.add(int(parent))

    leaf_subject_ids = all_ids - parent_ids
    return subject_name_by_id, leaf_subject_ids


def make_unique_skill_names(subject_name_by_id: dict[int, str], skill_ids: Iterable[int]) -> tuple[list[int], dict[int, str]]:
    ordered_skill_ids = sorted(set(skill_ids))
    used: set[str] = set()
    name_by_id: dict[int, str] = {}

    for sid in ordered_skill_ids:
        base = subject_name_by_id.get(sid, f"Subject_{sid}").strip() or f"Subject_{sid}"
        name = base
        if name in used:
            name = f"{base}__{sid}"
        used.add(name)
        name_by_id[sid] = name

    return ordered_skill_ids, name_by_id


def build_question_leaf_skill_map(question_csv: Path, leaf_subject_ids: set[int]) -> dict[int, set[int]]:
    question_to_leaf_skills: dict[int, set[int]] = {}
    with question_csv.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            qid = int(row["QuestionId"])
            chain = parse_subject_chain(row["SubjectId"])
            leaf_skills = {sid for sid in chain if sid in leaf_subject_ids}
            question_to_leaf_skills[qid] = leaf_skills
    return question_to_leaf_skills


def main() -> None:
    root = find_project_root()

    subject_csv = root / "data" / "metadata" / "subject_metadata.csv"
    question_csv = root / "data" / "metadata" / "question_metadata_task_1_2.csv"
    train_csv = root / "data" / "train_data" / "train_task_1_2.csv"
    output_csv = root / "data" / "train_task_1_2_leaf_skill_matrix.csv"

    subject_name_by_id, leaf_subject_ids = load_subject_metadata(subject_csv)
    question_leaf_map = build_question_leaf_skill_map(question_csv, leaf_subject_ids)

    all_used_leaf_skills: set[int] = set()
    for skills in question_leaf_map.values():
        all_used_leaf_skills.update(skills)

    ordered_skill_ids, skill_col_name_by_id = make_unique_skill_names(subject_name_by_id, all_used_leaf_skills)

    base_columns = ["QuestionId", "UserId", "AnswerId", "IsCorrect", "TimeTaken"]
    skill_columns = [skill_col_name_by_id[sid] for sid in ordered_skill_ids]
    output_columns = base_columns + skill_columns

    total_rows = count_data_rows(train_csv)
    update_every = max(total_rows // 200, 1)  # ~0.5% updates

    output_csv.parent.mkdir(parents=True, exist_ok=True)

    processed = 0
    missing_time_taken_reported = False

    with train_csv.open("r", encoding="utf-8-sig", newline="") as fin, output_csv.open(
        "w", encoding="utf-8", newline=""
    ) as fout:
        reader = csv.DictReader(fin)
        writer = csv.DictWriter(fout, fieldnames=output_columns)
        writer.writeheader()

        for row in reader:
            processed += 1
            qid = int(row["QuestionId"])
            q_leaf_skills = question_leaf_map.get(qid, set())

            if "TimeTaken" in row:
                time_taken = row.get("TimeTaken", "")
            else:
                time_taken = ""
                if not missing_time_taken_reported:
                    print("Info: 'TimeTaken' column not found in train CSV. Writing empty values.")
                    missing_time_taken_reported = True

            out = {
                "QuestionId": row.get("QuestionId", ""),
                "UserId": row.get("UserId", ""),
                "AnswerId": row.get("AnswerId", ""),
                "IsCorrect": row.get("IsCorrect", ""),
                "TimeTaken": time_taken,
            }

            for sid in ordered_skill_ids:
                out[skill_col_name_by_id[sid]] = 1 if sid in q_leaf_skills else 0

            writer.writerow(out)

            if processed == 1 or processed % update_every == 0 or processed == total_rows:
                print_progress(processed, total_rows, prefix="Building CSV")

    print(f"Finished. Output written to: {output_csv}")
    print(f"Rows written: {processed}")
    print(f"Leaf skill columns added: {len(skill_columns)}")


main()

Info: 'TimeTaken' column not found in train CSV. Writing empty values.
Building CSV: [########################################] 100.00% (15867850/15867850)
Finished. Output written to: /Users/atlasbuchholz/PycharmProjects/UBC/STAT_405_Project/data/train_task_1_2_leaf_skill_matrix.csv
Rows written: 15867850
Leaf skill columns added: 316


# now i want you to add the two more columns the gender of the student and the age of the student.
The gender is found in the data/metadata/student_metadata_task_** .csv and the date of birth can be found there. The age can then be inferred from the time of the users first answer found in data/metadata/answer_metadata_task_** .csv file. you can then also use this to readd the time taken to complete a problem

In [ ]:
# function here

This code extracts the data from the csv.

In [9]:
# %pip install pyarrow
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────────
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "data").exists():
    ROOT = ROOT.parent

INPUT_CSV = ROOT / "data" / "train_task_1_2_leaf_skill_matrix.csv"
OUTPUT_DIR = ROOT / "processing_data" / "output_parquet"
CHUNK_SIZE = 50_000
META_COLS = ["QuestionId", "UserId", "AnswerId", "IsCorrect", "TimeTaken"]
# ─────────────────────────────────────────────────────────────────────────────

if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_CSV}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Read header once to identify skill columns
header = pd.read_csv(INPUT_CSV, nrows=0).columns.tolist()
skill_cols = [c for c in header if c not in META_COLS]

print(f"Found {len(skill_cols)} skill columns")
print(f"Processing in chunks of {CHUNK_SIZE:,}...")

writer = None
total_rows = 0

for chunk_num, chunk in enumerate(
    pd.read_csv(INPUT_CSV, chunksize=CHUNK_SIZE, dtype={c: "int8" for c in skill_cols})
):
    # Pivot to long format, dropping zeros
    long = (
        chunk
        .melt(id_vars=META_COLS, value_vars=skill_cols, var_name="skill", value_name="value")
        .query("value != 0")
        .drop(columns="value")  # value is always 1, no need to store it
        .reset_index(drop=True)
    )

    # Cast to efficient types
    long["skill"] = long["skill"].astype("category")
    long["IsCorrect"] = long["IsCorrect"].astype("int8")

    # Write parquet chunk
    table = pa.Table.from_pandas(long)
    if writer is None:
        writer = pq.ParquetWriter(
            OUTPUT_DIR / "data.parquet",
            table.schema,
            compression="zstd",
            compression_level=9,
        )
    writer.write_table(table)

    total_rows += len(long)
    if chunk_num % 10 == 0:
        print(f"  Chunk {chunk_num:>4} | non-zero rows so far: {total_rows:>10,}")

if writer:
    writer.close()

print(f"\nDone. Total non-zero rows: {total_rows:,}")
print(f"Output: {OUTPUT_DIR / 'data.parquet'}")

# ── Check output size ─────────────────────────────────────────────────────────
size_mb = (OUTPUT_DIR / "data.parquet").stat().st_size / 1e6
print(f"File size: {size_mb:.1f} MB")

Found 316 skill columns
Processing in chunks of 50,000...
  Chunk    0 | non-zero rows so far:     54,663
  Chunk   10 | non-zero rows so far:    600,710
  Chunk   20 | non-zero rows so far:  1,146,482
  Chunk   30 | non-zero rows so far:  1,692,603
  Chunk   40 | non-zero rows so far:  2,239,264
  Chunk   50 | non-zero rows so far:  2,785,217
  Chunk   60 | non-zero rows so far:  3,331,156
  Chunk   70 | non-zero rows so far:  3,877,602
  Chunk   80 | non-zero rows so far:  4,423,702
  Chunk   90 | non-zero rows so far:  4,969,669
  Chunk  100 | non-zero rows so far:  5,516,098
  Chunk  110 | non-zero rows so far:  6,062,145
  Chunk  120 | non-zero rows so far:  6,608,908
  Chunk  130 | non-zero rows so far:  7,155,390
  Chunk  140 | non-zero rows so far:  7,701,674
  Chunk  150 | non-zero rows so far:  8,248,328
  Chunk  160 | non-zero rows so far:  8,794,766
  Chunk  170 | non-zero rows so far:  9,341,045
  Chunk  180 | non-zero rows so far:  9,887,306
  Chunk  190 | non-zero rows s